**Post-training rerun** of `yolo26s_smoke_fold0_640_s42_v1` (training cell replaced by loading the finished run).

# YOLO26s — Day-1 smoke run

COCO-pretrained YOLO26s, 640 px, fold 0, every fifth frame. Shared evaluation uses original images and COCO crowd annotations. The leaderboard metric remains unconfirmed.

## Environment

In [1]:
from pathlib import Path
import os
import subprocess
import sys

KAGGLE = Path('/kaggle/working').is_dir()
WORK = Path('/kaggle/working') if KAGGLE else Path.cwd()
REPO = WORK / 'ai-hw-object-detection' if KAGGLE else (WORK if (WORK / 'src').is_dir() else WORK.parent)
if not KAGGLE:
    WORK = REPO
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Stalex575/ai-hw-object-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
sys.path.insert(0, str(REPO))
for name in ('YOLO_CONFIG_DIR', 'MPLCONFIGDIR'):
    directory = WORK / 'outputs' / name.lower()
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(directory)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


eaea03eab3cefeb8068b0e7d00943684e495ce4f


## Data and run configuration

In [2]:
import json
import yaml
from src import evaluate, submit, yolo

roots = [REPO / 'data', REPO]
if KAGGLE:
    roots += [p.parent for p in Path('/kaggle/input').rglob('test_images.csv')]
DATA = next(p for p in roots if (p / 'train/annotations_coco.json').is_file())
PREP = WORK / 'prep' if KAGGLE else REPO / 'prep'
RUN_NAME = 'yolo26s_smoke_fold0_640_s42_v1'
OUT = WORK / 'outputs' / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
FOLD, FRAME_STEP = 0, 5
COMPARE_NMS = False
import logging
logger = logging.getLogger('hw2')
logger.setLevel(logging.INFO)
logger.propagate = False
for previous in list(logger.handlers):
    logger.removeHandler(previous)
    previous.close()
file_handler = logging.FileHandler(OUT / 'run.log')
file_handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
logger.addHandler(file_handler)
logger.info('Data=%s; fold=%s; frame_step=%s', DATA, FOLD, FRAME_STEP)
print('Data:', DATA, '\nRun:', OUT)


Data: /home/stalex/unik_shit/ai/ai-hw-object-detection 
Run: /home/stalex/unik_shit/ai/ai-hw-object-detection/outputs/yolo26s_smoke_fold0_640_s42_v1


## Fold preparation and evaluator check

In [3]:
DATA_YAML, GT_PATH, fold_info = yolo.prepare_fold(DATA, PREP, REPO / 'folds.csv', FOLD, FRAME_STEP)
gt = evaluate.read_json(GT_PATH)
oracle = evaluate.evaluate_predictions(gt, evaluate.oracle_predictions(gt))
assert oracle['metrics']['mAP'] > 0.999
print(fold_info)
print('GT sanity check:', oracle['metrics'])
evaluate.write_json(OUT / 'preflight.json', {**fold_info, 'oracle': oracle['metrics']})

logger.info('Preflight: %s; oracle: %s', fold_info, oracle['metrics'])


{'train_images': 3243, 'val_images': 1625, 'fold': 0, 'frame_step': 5}
GT sanity check: {'mAP': 1.0, 'AP50': 1.0, 'AP75': 1.0, 'AP_small': 1.0, 'AP_medium': 0.9999985609330542, 'AP_large': 1.0}


## Training

In [4]:
# POST-ONLY RERUN: training already finished in the first execution (26 min); the post-training
# cells crashed on a zero-area box (fixed in src/yolo.py). Load the finished run instead of retraining.
from ultralytics import YOLO
recipe = yaml.safe_load((REPO / 'configs/yolo26_smoke.yaml').read_text())
runtime = json.loads((OUT / 'runtime.json').read_text())
info = json.loads((OUT / 'training.json').read_text())
BEST, training_hours = Path(info['best']), info['hours']
print('Loaded', BEST, info)


Loaded /home/stalex/unik_shit/ai/ai-hw-object-detection/outputs/yolo26s_smoke_fold0_640_s42_v1/train/weights/best.pt {'hours': 0.4136920305788889, 'best': '/home/stalex/unik_shit/ai/ai-hw-object-detection/outputs/yolo26s_smoke_fold0_640_s42_v1/train/weights/best.pt', 'epochs': 20}


## Shared validation and inference heads

In [5]:
predictions = yolo.predict(YOLO(BEST), gt['images'], DATA / 'train', OUT / 'val_predictions.json', imgsz=recipe['imgsz'])
metrics = evaluate.evaluate_predictions(gt, predictions)
evaluate.write_json(OUT / 'metrics.json', metrics)
print(metrics['metric_status'])
print(metrics['metrics'])
if COMPARE_NMS:
    nms_predictions = yolo.predict(YOLO(BEST), gt['images'], DATA / 'train', OUT / 'val_predictions_nms.json',
                                   imgsz=recipe['imgsz'], end2end=False)
    nms_metrics = evaluate.evaluate_predictions(gt, nms_predictions)
    evaluate.write_json(OUT / 'metrics_nms.json', nms_metrics)
logger.info('Shared validation metrics: %s', metrics['metrics'])


COCO diagnostics; competition scorer parity unconfirmed
{'mAP': 0.18729915348598852, 'AP50': 0.3429138770253952, 'AP75': 0.18305999735659004, 'AP_small': 0.04126924351544389, 'AP_medium': 0.1285953449352984, 'AP_large': 0.30777085609224214}


## Validation examples and batch-one speed

In [6]:
from PIL import Image, ImageDraw
from src.viz_gt import grid

examples, clips = [], set()
for image in gt['images']:
    clip = Path(image['file_name']).parent.name
    if clip in clips:
        continue
    clips.add(clip)
    picture = Image.open(DATA / 'train' / image['file_name']).convert('RGB')
    draw = ImageDraw.Draw(picture)
    targets = [a for a in gt['annotations'] if a['image_id'] == image['id'] and not a.get('iscrowd')]
    detected = [p for p in predictions if p['image_id'] == image['id'] and p['score'] >= 0.1]
    for records, color in ((targets, 'lime'), (detected, 'red')):
        for record in records:
            x, y, w, h = record['bbox']
            draw.rectangle((x, y, x + w, y + h), outline=color, width=3)
    draw.text((10, 10), clip + ' | green: GT, red: predictions', fill='white', stroke_width=2, stroke_fill='black')
    examples.append(picture)
    if len(examples) == 4:
        break
grid(examples, cols=2, tile_w=640).save(OUT / 'validation_examples.jpg', quality=90)
paths = [DATA / 'train' / image['file_name'] for image in gt['images']]
sample = [paths[round(i * (len(paths) - 1) / 31)] for i in range(32)]
speed = yolo.benchmark(YOLO(BEST), sample, imgsz=recipe['imgsz'])
evaluate.write_json(OUT / 'speed.json', speed)
logger.info('Speed: %s', speed)
print(speed)


{'fps': 51.36665920300969, 'mean_ms': 19.46788083001138, 'batch': 1, 'warmup': 20, 'repeats': 100, 'imgsz': 640, 'half': True, 'end2end': True, 'device': '0', 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU', 'scope': 'image decode, preprocessing, inference, post-processing and CPU box transfer'}


## Test predictions and submission

In [7]:
test_images = submit.read_test_images(DATA / 'test_images.csv')
test_predictions = yolo.predict(YOLO(BEST), test_images, DATA, OUT / 'test_predictions.json', imgsz=recipe['imgsz'])
submit.write_submission(test_images, test_predictions, OUT / 'submission.csv')
submission_check = submit.validate_submission(test_images, OUT / 'submission.csv')
assert submission_check['detections'] > 0
evaluate.write_json(OUT / 'submission_check.json', submission_check)
print(submission_check)
logger.info('Submission check: %s', submission_check)


{'images': 3811, 'images_with_predictions': 3811, 'detections': 1762171}


## Run artifacts

In [8]:
import csv
import zipfile
import shutil

row = {'id': RUN_NAME, 'owner': 'Oleh', 'family': 'YOLO26s', 'fold': FOLD,
       **metrics['metrics'], 'FPS': speed['fps'], 'GPU_h_training': training_hours,
       'public_LB': '', 'commit': runtime['commit']}
with (OUT / 'experiment.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)
shutil.copy2(OUT / 'submission.csv', OUT.parent / f'{RUN_NAME}_submission.csv')
archive = OUT.parent / f'{RUN_NAME}_results.zip'
logger.info('Run complete; results archive=%s', archive)
file_handler.flush()
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in OUT.rglob('*'):
        if path.is_file() and not (path.name.startswith('epoch') and path.suffix == '.pt'):
            bundle.write(path, path.relative_to(OUT))
print('Submission:', OUT.parent / f'{RUN_NAME}_submission.csv')
print('Run archive:', archive)


Submission: /home/stalex/unik_shit/ai/ai-hw-object-detection/outputs/yolo26s_smoke_fold0_640_s42_v1_submission.csv
Run archive: /home/stalex/unik_shit/ai/ai-hw-object-detection/outputs/yolo26s_smoke_fold0_640_s42_v1_results.zip
